# Extração de dados via pyalex

In [ ]:
# Insira aqui o email cadastrado no OpenAlex
pyalex.config.email = ""

if "TARGET_CONCEPT_ID" not in globals() or TARGET_CONCEPT_ID is None:
    TARGET_CONCEPT_ID = "XXXXXXXXXXX"  # fallback: só é usado se a validação da célula anterior tiver falhado
    
YEAR_START = 0000
YEAR_END = 0000

# String de busca modelada como Dicionário
KEYWORDS_AREAS = {
    # Organizado em ("ÁREA 1":["PALAVRA-CHAVE 1","PALAVRA-CHAVE 2"])
    "A":["B","C","D","E"],
    "X":["Y","Z"],
}

# Dicionários para a coleta de Artigos/IDs
unique_works = {}
id_to_categories = {}

print(f"--- INICIANDO EXTRAÇÃO GLOBAL (A partir de {YEAR_START} até {YEAR_END}) ---")
print(f"Filtros aplicados: Idioma = Inglês | Conceito = {top_concept["display_name"]} | Tipo = Artigo\n")

# Estrutura de repetição que itera sobre o conteúdo do dicionário KEYWORDS_AREAS
for category, terms in KEYWORDS_AREAS.items():
    print(f"Categoria: {category}")
    for term in terms:
        print(f"  > Buscando termo: {term}...", end=" ")
        try:
            # Filtros aplicados na consulta dos artigos e busca da palavra-chave no título e resumo do trabalho
            query = (
                Works()
                .filter(concepts={"id": TARGET_CONCEPT_ID},
                        from_publication_date=f"{YEAR_START}-01-01",
                        to_publication_date=f"{YEAR_END}-12-31",
                        type="article",
                        language="en")  # ← FILTRO DE IDIOMA INGLÊS
                .filter(title_and_abstract={"search": term})
            )

            # O .paginate() percorre todas as páginas automaticamente
            # Encadeamento de estruturas que impede a duplicidade de artigos
            count_termo = 0
            for page in query.paginate(per_page=200): #per_page=200 é o limite de artigos retornado pela API
                for work in page:
                    w_id = work['id']
                    if w_id not in unique_works:
                        unique_works[w_id] = work
                    # Classificação múltipla de artigos
                    if w_id not in id_to_categories:
                        id_to_categories[w_id] = set()
                    id_to_categories[w_id].add(category)
                    count_termo += 1

            print(f"Encontrados: {count_termo}")

        except Exception as e:
            print(f"Erro ao buscar '{term}': {e}")

print(f"\nExtração concluída: {len(unique_works)} artigos únicos encontrados (apenas em inglês).")
